In [2]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer

In [3]:
# file paths
home = '/store/carroll/sbgplants/'
ref = os.path.join(home, 'schema')
raw = os.path.join(home, 'data', 'raw')

doi = os.path.join(raw, '10.15485.1618130') # Locations, metadata, and species cover from field sampling survey associated with NEON AOP survey, East River, CO 2018

out_folder = os.path.join(home, 'data', 'out_csv')

table = 'insitu_plot_event'

In [4]:
# load schema and dtype
schema = pd.read_csv(os.path.join(ref, 'sbgplants-schema.csv'))
data_types = pd.read_csv(os.path.join(ref, 'data-types.csv'))

# view relevant schema
schema = schema[schema.table_name==table]
schema

,table_name,column_name,data_type
25,insitu_plot_event,collection_date,date
26,insitu_plot_event,plot_name,character
27,insitu_plot_event,geom,USER-DEFINED
28,insitu_plot_event,insitu_plot_id,uuid


In [5]:
# no relevant data-types info 

In [5]:
# load relevant output tables
plot = pd.read_csv(os.path.join(out_folder, 'plot.csv'))
plot_event_metadata = pd.read_csv(os.path.join(out_folder, 'plot_event_metadata.csv'))
plot_event_metadata = plot_event_metadata[['plot_name', 'collection_date']]
plot_event_metadata

,plot_name,collection_date
0,001-ER18,2018-06-14
1,002-ER18,2018-06-14
2,003-ER18,2018-06-14
3,004-ER18,2018-06-14
4,005-ER18,2018-06-14
...,...,...
472,474-ER18,2018-07-30
473,475-ER18,2018-07-30
474,476-ER18,2018-07-30
475,477-ER18,2018-07-30


In [6]:
# load, update relevant raw tables
dat = pd.read_csv(os.path.join(doi, 'raw_rtk_gps_points.csv'))

# drop rows where SampleSiteCode (plot name) is NA
dat = dat[dat['SampleSiteCode'].notna()]

# transform to 32613
transformer = Transformer.from_crs('EPSG:26913', 'EPSG:32613', always_xy=True)
dat['Easting'], dat['Northing'] = transformer.transform(dat['Easting'].values, dat['Northing'].values)

# prepare geometry
gdf = gpd.GeoDataFrame(
    dat,
    geometry=[Point(xy) for xy in zip(dat['Easting'], dat['Northing'])],
    crs='EPSG:32613'
)
gdf['geometry_wkt'] = gdf['geometry'].apply(lambda geom: geom.wkt)

gdf

,SampleSiteCode,FieldPointName,Northing,Easting,Elevation,Code,EPSG,Unnamed: 7,Unnamed: 8,Unnamed: 9,geometry,geometry_wkt
0,001-ER18,001-A,4.313906e+06,327909.986803,2908.642,CP-PLOTS,26913,NaN,NaN,NaN,POINT (327909.987 4313905.668),POINT (327909.98680344573 4313905.668121463)
1,001-ER18,001-B,4.313906e+06,327909.046803,2908.522,CP-PLOTS,26913,NaN,NaN,NaN,POINT (327909.047 4313905.802),POINT (327909.0468034833 4313905.8021165095)
2,001-ER18,001-C,4.313907e+06,327909.158801,2908.548,CP-PLOTS,26913,NaN,NaN,NaN,POINT (327909.159 4313906.75),POINT (327909.15880066797 4313906.750119842)
3,001-ER18,001-D,4.313907e+06,327910.084801,2908.632,CP-PLOTS,26913,NaN,NaN,NaN,POINT (327910.085 4313906.58),POINT (327910.0848007422 4313906.580124612)
4,002-ER18,002-A,4.313912e+06,327909.986785,2908.659,CP-PLOTS,26913,NaN,NaN,NaN,POINT (327909.987 4313911.825),POINT (327909.9867854939 4313911.825138976)
...,...,...,...,...,...,...,...,...,...,...,...,...
1017,434-ER18,TR-434,4.316230e+06,321249.281776,3054.656,CP-PLOT,26913,NaN,NaN,NaN,POINT (321249.282 4316230.427),POINT (321249.28177596896 4316230.427414311)
1018,435-ER18,TR-435,4.316239e+06,321265.987795,3061.681,CP-PLOT,26913,NaN,NaN,NaN,POINT (321265.988 4316238.952),POINT (321265.98779536283 4316238.952468981)
1019,436-ER18,TR-436,4.316202e+06,321201.216731,3027.257,CP-PLOT,26913,NaN,NaN,NaN,POINT (321201.217 4316201.997),POINT (321201.2167305222 4316201.997247643)
1020,437-ER18,TR-437,4.316231e+06,321135.566486,2996.520,CP-PLOT,26913,NaN,NaN,NaN,POINT (321135.566 4316231.293),POINT (321135.56648641464 4316231.293184906)


In [7]:
# prepare & populate out table
out_table = pd.DataFrame(columns=schema['column_name'].unique())

out_table['plot_name'] = gdf['SampleSiteCode']
out_table['geom'] = gdf['geometry_wkt']

# merge collection date
out_table = out_table.drop(columns=['collection_date'])
out_table = pd.merge(out_table, plot_event_metadata, on='plot_name', how='left', suffixes=('',''))

# reorder columns
out_table = out_table[schema.column_name]

out_table

,collection_date,plot_name,geom,insitu_plot_id
0,2018-06-14,001-ER18,POINT (327909.98680344573 4313905.668121463),NaN
1,2018-06-14,001-ER18,POINT (327909.0468034833 4313905.8021165095),NaN
2,2018-06-14,001-ER18,POINT (327909.15880066797 4313906.750119842),NaN
3,2018-06-14,001-ER18,POINT (327910.0848007422 4313906.580124612),NaN
4,2018-06-14,002-ER18,POINT (327909.9867854939 4313911.825138976),NaN
...,...,...,...,...
1017,2018-06-28,434-ER18,POINT (321249.28177596896 4316230.427414311),NaN
1018,2018-06-28,435-ER18,POINT (321265.98779536283 4316238.952468981),NaN
1019,2018-06-28,436-ER18,POINT (321201.2167305222 4316201.997247643),NaN
1020,2018-06-28,437-ER18,POINT (321135.56648641464 4316231.293184906),NaN


In [8]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)